# Premier League Match Prediction: Part 3 - Match Predictor & Live API Inference

Welcome to the production inference engine. This notebook uses the trained ensemble from **Part 2** to deliver comprehensive match analytics.

### What This Predictor Delivers:
1. **Pre-Match 'Tale of the Tape'**: Compares the two clubs by Current Elo, Recent Form Points, Goals Scored/Conceded, and Shots on Target.
2. **Calibrated Outcome Probabilities**: Displays percentage probabilities for Home Win, Draw, and Away Win.
3. **Expected Goals (xG)**: Forecasts expected goals for both teams (e.g., $1.85$ vs $0.92$).
4. **Exact Scoreline Probabilities**: Generates a bivariate Poisson score grid and ranks the top 5 most likely exact final scores (e.g. 2-1, 1-1, 2-0).
5. **Live Automated Fixture Predictions**: Connects directly to `football-data.org` to fetch upcoming Premier League fixtures and forecast the entire game-week automatically.

### Step 1: Imports & Setup
We load libraries for prediction, formatting, API requests, and model inference.

In [11]:
import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import requests
from scipy.stats import poisson

print("Inference environment initialized successfully!")

Inference environment initialized successfully!


### Step 2: Load Processed Match History & Extract Latest Team States
We load `dataset/processed_match_dataset.csv` and extract each club's most recent statistics (current Elo, last 5 games form, rolling goal averages, shots on target, and venue-specific form).

In [12]:
data_path = Path('dataset/processed_match_dataset.csv')
if not data_path.exists():
    raise FileNotFoundError("dataset/processed_match_dataset.csv not found! Please run 01_Data_Prep_and_Feature_Engineering.ipynb first.")

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])

# Get all unique clubs
all_teams = sorted(list(set(df['HomeTeam']).union(set(df['AwayTeam']))))

# Build latest team snapshot dictionary
latest_team_stats = {}

for team in all_teams:
    # Get all matches involving this team (sorted chronologically)
    team_matches = df[(df['HomeTeam'] == team) | (df['AwayTeam'] == team)].sort_values('Date')
    if len(team_matches) == 0:
        continue
    
    last_match = team_matches.iloc[-1]
    is_home = (last_match['HomeTeam'] == team)
    
    # Venue-specific matches
    home_matches = df[df['HomeTeam'] == team].sort_values('Date')
    away_matches = df[df['AwayTeam'] == team].sort_values('Date')
    venue_home = home_matches.iloc[-1]['Home_Venue_Points_5'] if len(home_matches) > 0 else 6.0
    venue_away = away_matches.iloc[-1]['Away_Venue_Points_5'] if len(away_matches) > 0 else 6.0
    
    # Extract latest stats
    elo = last_match['Home_Elo'] if is_home else last_match['Away_Elo']
    form_5 = last_match['Home_Roll_Points_5'] if is_home else last_match['Away_Roll_Points_5']
    gf_5 = last_match['Home_Roll_GF_5'] if is_home else last_match['Away_Roll_GF_5']
    ga_5 = last_match['Home_Roll_GA_5'] if is_home else last_match['Away_Roll_GA_5']
    gd_5 = last_match['Home_Roll_GD_5'] if is_home else last_match['Away_Roll_GD_5']
    stf_5 = last_match['Home_Roll_STF_5'] if is_home else last_match['Away_Roll_STF_5']
    sta_5 = last_match['Home_Roll_STA_5'] if is_home else last_match['Away_Roll_STA_5']
    stratio_5 = last_match['Home_Roll_STRatio_5'] if is_home else last_match['Away_Roll_STRatio_5']
    
    latest_team_stats[team] = {
        'Elo': round(float(elo), 1),
        'Form_5': round(float(form_5), 1),
        'GF_5': round(float(gf_5), 2),
        'GA_5': round(float(ga_5), 2),
        'GD_5': round(float(gd_5), 2),
        'STF_5': round(float(stf_5), 2),
        'STA_5': round(float(sta_5), 2),
        'STRatio_5': round(float(stratio_5), 2),
        'Venue_Home': round(float(venue_home), 1),
        'Venue_Away': round(float(venue_away), 1),
        'Last_Date': last_match['Date']
    }

print(f"Extracted latest statistical snapshots for {len(latest_team_stats)} Premier League clubs.")
# Display Top 10 ranked clubs by Elo
elo_ranking = pd.DataFrame.from_dict(latest_team_stats, orient='index').sort_values('Elo', ascending=False)
elo_ranking[['Elo', 'Form_5', 'GF_5', 'GA_5', 'STF_5']].head(10)

Extracted latest statistical snapshots for 27 Premier League clubs.


,Elo,Form_5,GF_5,GA_5,STF_5
Arsenal,1787.0,9.0,1.6,0.8,5.2
Man City,1781.8,11.0,2.0,1.0,6.4
Liverpool,1681.0,9.0,2.0,1.4,4.8
Man United,1648.5,10.0,1.8,1.4,5.4
Bournemouth,1617.5,11.0,2.2,1.2,4.6
Brighton,1609.6,10.0,2.0,1.2,5.8
Aston Villa,1591.1,7.0,1.6,1.4,4.2
Brentford,1583.2,6.0,1.2,0.8,4.0
Nott'm Forest,1581.9,13.0,3.2,0.6,5.0
Everton,1568.8,5.0,2.0,1.8,5.8


### Step 3: Define Poisson Goal Model & Load Model Artifacts
In this cell, we define `PoissonGoalModel` and fit it directly from `df` (which takes ~0.01s). This completely eliminates cross-notebook pickle `AttributeError` issues.
We also load the trained `calibrated_xgb` model and `feature_cols` from `saved_models/`.

In [13]:
class PoissonGoalModel:
    """Bivariate Poisson model for football match outcome and goal expectations."""
    def __init__(self):
        self.attack = {}
        self.defense = {}
        self.home_adv = 0.25
        self.league_avg = 1.35

    def fit(self, matches_df):
        home_goals = matches_df.groupby('HomeTeam')['FTHG'].agg(['sum', 'count'])
        away_goals = matches_df.groupby('AwayTeam')['FTAG'].agg(['sum', 'count'])
        home_conceded = matches_df.groupby('HomeTeam')['FTAG'].agg(['sum', 'count'])
        away_conceded = matches_df.groupby('AwayTeam')['FTHG'].agg(['sum', 'count'])
        
        all_teams = sorted(list(set(matches_df['HomeTeam']).union(set(matches_df['AwayTeam']))))
        total_games = len(matches_df)
        total_home_goals = matches_df['FTHG'].sum()
        total_away_goals = matches_df['FTAG'].sum()
        
        avg_home_scored = total_home_goals / total_games
        avg_away_scored = total_away_goals / total_games
        self.home_adv = np.log(avg_home_scored / avg_away_scored)
        self.league_avg = (total_home_goals + total_away_goals) / (2.0 * total_games)
        
        for t in all_teams:
            scored = (home_goals.loc[t, 'sum'] if t in home_goals.index else 0) + (away_goals.loc[t, 'sum'] if t in away_goals.index else 0)
            games = (home_goals.loc[t, 'count'] if t in home_goals.index else 0) + (away_goals.loc[t, 'count'] if t in away_goals.index else 0)
            conceded = (home_conceded.loc[t, 'sum'] if t in home_conceded.index else 0) + (away_conceded.loc[t, 'sum'] if t in away_conceded.index else 0)
            
            t_avg_scored = (scored / games) if games > 0 else self.league_avg
            t_avg_conceded = (conceded / games) if games > 0 else self.league_avg
            
            self.attack[t] = t_avg_scored / self.league_avg
            self.defense[t] = t_avg_conceded / self.league_avg

    def predict_lambdas(self, home_team, away_team):
        att_h = self.attack.get(home_team, 1.0)
        def_h = self.defense.get(home_team, 1.0)
        att_a = self.attack.get(away_team, 1.0)
        def_a = self.defense.get(away_team, 1.0)
        
        lambda_home = att_h * def_a * self.league_avg * np.exp(self.home_adv / 2.0)
        lambda_away = att_a * def_h * self.league_avg * np.exp(-self.home_adv / 2.0)
        return max(float(lambda_home), 0.2), max(float(lambda_away), 0.2)

    def predict_match_probs(self, home_team, away_team, max_goals=8):
        lh, la = self.predict_lambdas(home_team, away_team)
        p_home_goals = [poisson.pmf(i, lh) for i in range(max_goals)]
        p_away_goals = [poisson.pmf(j, la) for j in range(max_goals)]
        
        matrix = np.outer(p_home_goals, p_away_goals)
        p_home_win = np.sum(np.tril(matrix, -1))
        p_draw = np.sum(np.diag(matrix))
        p_away_win = np.sum(np.triu(matrix, 1))
        
        total = p_home_win + p_draw + p_away_win
        return np.array([p_home_win / total, p_draw / total, p_away_win / total])

# 1. Fit Poisson Model directly on match history
poisson_model = PoissonGoalModel()
poisson_model.fit(df)
print("Poisson Goal Model initialized and fitted on match data.")

# 2. Load trained XGBoost and feature column names
models_dir = Path('saved_models')
xgb_path = models_dir / 'calibrated_xgb.pkl'
features_path = models_dir / 'feature_cols.pkl'

if xgb_path.exists() and features_path.exists():
    with open(xgb_path, 'rb') as f:
        calibrated_xgb = pickle.load(f)
    with open(features_path, 'rb') as f:
        feature_cols = pickle.load(f)
    print("Loaded calibrated_xgb and feature_cols from saved_models/")
else:
    # Fallback in case notebook 2 hasn't been saved yet
    print("Artifacts not found in saved_models/. Fitting Calibrated XGBoost directly...")
    from xgboost import XGBClassifier
    from sklearn.calibration import CalibratedClassifierCV
    feature_cols = [
        'Home_Elo', 'Away_Elo', 'Elo_Diff',
        'Home_Roll_Points_5', 'Away_Roll_Points_5', 'Diff_Form',
        'Home_Roll_GF_5', 'Away_Roll_GF_5', 'Home_Roll_GA_5', 'Away_Roll_GA_5', 'Diff_Goals', 'Diff_GD',
        'Home_Roll_STF_5', 'Away_Roll_STF_5', 'Home_Roll_STA_5', 'Away_Roll_STA_5', 'Diff_ST',
        'Home_Roll_STRatio_5', 'Away_Roll_STRatio_5', 'Diff_STRatio',
        'Home_Venue_Points_5', 'Away_Venue_Points_5', 'Diff_Venue_Strength',
        'Home_Rest_Days', 'Away_Rest_Days', 'Diff_Rest_Days'
    ]
    xgb_base = XGBClassifier(max_depth=3, learning_rate=0.03, n_estimators=100, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, random_state=42, eval_metric='mlogloss')
    calibrated_xgb = CalibratedClassifierCV(estimator=xgb_base, method='sigmoid', cv=3)
    calibrated_xgb.fit(df[feature_cols].iloc[50:], df['Target'].values[50:])
    print("Trained fresh Calibrated XGBoost model.")

Poisson Goal Model initialized and fitted on match data.
Loaded calibrated_xgb and feature_cols from saved_models/


### Step 4: The Match Prediction Engine (`predict_fixture`)
This function forecasts any Premier League match using the blended ensemble (60% XGBoost + 40% Poisson).
It outputs:
- **Pre-match Tale of the Tape**
- **Expected Goals (xG)**
- **Calibrated Outcome Probabilities** with visual bars
- **Top 5 Most Likely Exact Final Scores**

In [14]:
def predict_fixture(home_team, away_team, rest_days_home=7, rest_days_away=7):
    """
    Predicts match outcome, probabilities, xG, and top scorelines between home_team and away_team.
    """
    if home_team not in latest_team_stats or away_team not in latest_team_stats:
        missing = [t for t in [home_team, away_team] if t not in latest_team_stats]
        raise ValueError(f"Team(s) not found in dataset: {missing}. Available clubs:\n{list(latest_team_stats.keys())}")
        
    h = latest_team_stats[home_team]
    a = latest_team_stats[away_team]
    
    # 1. Construct differential feature vector
    feat_dict = {
        'Home_Elo': h['Elo'],
        'Away_Elo': a['Elo'],
        'Elo_Diff': h['Elo'] - a['Elo'],
        'Home_Roll_Points_5': h['Form_5'],
        'Away_Roll_Points_5': a['Form_5'],
        'Diff_Form': h['Form_5'] - a['Form_5'],
        'Home_Roll_GF_5': h['GF_5'],
        'Away_Roll_GF_5': a['GF_5'],
        'Home_Roll_GA_5': h['GA_5'],
        'Away_Roll_GA_5': a['GA_5'],
        'Diff_Goals': h['GF_5'] - a['GA_5'],
        'Diff_GD': h['GD_5'] - a['GD_5'],
        'Home_Roll_STF_5': h['STF_5'],
        'Away_Roll_STF_5': a['STF_5'],
        'Home_Roll_STA_5': h['STA_5'],
        'Away_Roll_STA_5': a['STA_5'],
        'Diff_ST': h['STF_5'] - a['STA_5'],
        'Home_Roll_STRatio_5': h['STRatio_5'],
        'Away_Roll_STRatio_5': a['STRatio_5'],
        'Diff_STRatio': h['STRatio_5'] - a['STRatio_5'],
        'Home_Venue_Points_5': h['Venue_Home'],
        'Away_Venue_Points_5': a['Venue_Away'],
        'Diff_Venue_Strength': h['Venue_Home'] - a['Venue_Away'],
        'Home_Rest_Days': rest_days_home,
        'Away_Rest_Days': rest_days_away,
        'Diff_Rest_Days': rest_days_home - rest_days_away
    }
    
    feat_df = pd.DataFrame([feat_dict])[feature_cols]
    
    # 2. Model Predictions
    xgb_p = calibrated_xgb.predict_proba(feat_df)[0]
    poisson_p = poisson_model.predict_match_probs(home_team, away_team)
    
    # Ensemble blend (60% XGBoost + 40% Poisson)
    final_p = 0.60 * xgb_p + 0.40 * poisson_p
    
    # 3. Poisson Expected Goals (xG) and Score Grid
    lh, la = poisson_model.predict_lambdas(home_team, away_team)
    scores = []
    for i in range(6):
        for j in range(6):
            p_score = poisson.pmf(i, lh) * poisson.pmf(j, la)
            scores.append((f"{i} - {j}", p_score * 100.0))
    scores.sort(key=lambda x: x[1], reverse=True)
    
    # 4. Formatted Display
    print("=" * 65)
    print(f"MATCH FORECAST: {home_team.upper()} (Home) vs {away_team.upper()} (Away)")
    print("=" * 65)
    
    print("\n--- TALE OF THE TAPE ---")
    comparison = pd.DataFrame({
        'Metric': ['Club Elo Rating', 'Recent Form (Pts / 15)', 'Avg Goals Scored', 'Avg Goals Conceded', 'Shots on Target / Match'],
        home_team: [h['Elo'], h['Form_5'], h['GF_5'], h['GA_5'], h['STF_5']],
        away_team: [a['Elo'], a['Form_5'], a['GF_5'], a['GA_5'], a['STF_5']]
    })
    print(comparison.to_string(index=False))
    
    print(f"\n--- EXPECTED GOALS (xG) ---")
    print(f"{home_team}: {lh:.2f} xG   |   {away_team}: {la:.2f} xG")
    
    print("\n--- OUTCOME PROBABILITIES ---")
    print(f"{home_team} Win:  {final_p[0] * 100:5.1f}%  " + "█" * int(round(final_p[0] * 25)))
    print(f"Draw:             {final_p[1] * 100:5.1f}%  " + "█" * int(round(final_p[1] * 25)))
    print(f"{away_team} Win:  {final_p[2] * 100:5.1f}%  " + "█" * int(round(final_p[2] * 25)))
    
    print("\n--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---")
    for score, prob in scores[:5]:
        print(f"  {score:<8}: {prob:4.1f}%")
    print("=" * 65)
    
    return {
        'Home_Win_Prob': final_p[0],
        'Draw_Prob': final_p[1],
        'Away_Win_Prob': final_p[2],
        'Expected_Home_Goals': lh,
        'Expected_Away_Goals': la,
        'Top_Scores': scores[:5]
    }

print("predict_fixture function defined successfully!")

predict_fixture function defined successfully!


### Step 5: Test Run Match Predictions
Let's predict two matchups:
1. `Liverpool` vs `Brentford`
2. `Arsenal` vs `Man City`

In [15]:
res1 = predict_fixture('Liverpool', 'Brentford')

MATCH FORECAST: LIVERPOOL (Home) vs BRENTFORD (Away)

--- TALE OF THE TAPE ---
                 Metric  Liverpool  Brentford
        Club Elo Rating     1681.0     1583.2
 Recent Form (Pts / 15)        9.0        6.0
       Avg Goals Scored        2.0        1.2
     Avg Goals Conceded        1.4        0.8
Shots on Target / Match        4.8        4.0

--- EXPECTED GOALS (xG) ---
Liverpool: 2.32 xG   |   Brentford: 1.00 xG

--- OUTCOME PROBABILITIES ---
Liverpool Win:   62.9%  ████████████████
Draw:              20.5%  █████
Brentford Win:   16.5%  ████

--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---
  2 - 1   :  9.7%
  2 - 0   :  9.7%
  1 - 1   :  8.4%
  1 - 0   :  8.4%
  3 - 1   :  7.5%


In [16]:
res2 = predict_fixture('Arsenal', 'Man City')

MATCH FORECAST: ARSENAL (Home) vs MAN CITY (Away)

--- TALE OF THE TAPE ---
                 Metric  Arsenal  Man City
        Club Elo Rating   1787.0    1781.8
 Recent Form (Pts / 15)      9.0      11.0
       Avg Goals Scored      1.6       2.0
     Avg Goals Conceded      0.8       1.0
Shots on Target / Match      5.2       6.4

--- EXPECTED GOALS (xG) ---
Arsenal: 1.36 xG   |   Man City: 1.38 xG

--- OUTCOME PROBABILITIES ---
Arsenal Win:   43.0%  ███████████
Draw:              26.6%  ███████
Man City Win:   30.4%  ████████

--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---
  1 - 1   : 12.1%
  0 - 1   :  8.9%
  1 - 0   :  8.8%
  1 - 2   :  8.4%
  2 - 1   :  8.2%


### Step 6: Automated Live Fixture Predictor via football-data.org API
In this cell, we query `football-data.org` to automatically fetch scheduled Premier League fixtures (`status=SCHEDULED`), match team names to our database, and run predictions for the upcoming fixtures without manual typing.

In [17]:
api_token = '60e60f6073da4ebf97760ff585ef0432'
url = 'https://api.football-data.org/v4/competitions/PL/matches?status=SCHEDULED'
headers = {'X-Auth-Token': api_token}

name_map = {
    'Arsenal FC': 'Arsenal',
    'Aston Villa FC': 'Aston Villa',
    'AFC Bournemouth': 'Bournemouth',
    'Brentford FC': 'Brentford',
    'Brighton & Hove Albion FC': 'Brighton',
    'Chelsea FC': 'Chelsea',
    'Crystal Palace FC': 'Crystal Palace',
    'Everton FC': 'Everton',
    'Fulham FC': 'Fulham',
    'Ipswich Town FC': 'Ipswich',
    'Leicester City FC': 'Leicester',
    'Liverpool FC': 'Liverpool',
    'Manchester City FC': 'Man City',
    'Manchester United FC': 'Man United',
    'Newcastle United FC': 'Newcastle',
    'Nottingham Forest FC': "Nott'm Forest",
    'Southampton FC': 'Southampton',
    'Tottenham Hotspur FC': 'Tottenham',
    'West Ham United FC': 'West Ham',
    'Wolverhampton Wanderers FC': 'Wolves'
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    if response.status_code == 200:
        matches = response.json().get('matches', [])
        print(f"Retrieved {len(matches)} scheduled fixtures from football-data.org\n")
        
        predicted_count = 0
        for m in matches[:5]:
            raw_h = m['homeTeam']['name']
            raw_a = m['awayTeam']['name']
            h_team = name_map.get(raw_h, raw_h)
            a_team = name_map.get(raw_a, raw_a)
            
            if h_team in latest_team_stats and a_team in latest_team_stats:
                predict_fixture(h_team, a_team)
                predicted_count += 1
                print("\n")
            else:
                print(f"Skipping {raw_h} vs {raw_a}: Not found in active snapshot.")
        
        if predicted_count == 0 and len(matches) == 0:
            print("No matches currently in 'SCHEDULED' status. Running demo marquee matchup:")
            predict_fixture('Chelsea', 'Tottenham')
    else:
        print(f"API status {response.status_code}: {response.text}")
        print("Running fallback marquee matchup:")
        predict_fixture('Chelsea', 'Tottenham')
except Exception as e:
    print(f"Could not reach API ({e}). Running fallback demo matchup:")
    predict_fixture('Chelsea', 'Tottenham')

Retrieved 330 scheduled fixtures from football-data.org

Skipping Arsenal FC vs Leeds United FC: Not found in active snapshot.
Skipping Sunderland AFC vs Brighton & Hove Albion FC: Not found in active snapshot.
MATCH FORECAST: CHELSEA (Home) vs BOURNEMOUTH (Away)

--- TALE OF THE TAPE ---
                 Metric  Chelsea  Bournemouth
        Club Elo Rating   1561.6       1617.5
 Recent Form (Pts / 15)      0.0         11.0
       Avg Goals Scored      0.2          2.2
     Avg Goals Conceded      2.6          1.2
Shots on Target / Match      3.0          4.6

--- EXPECTED GOALS (xG) ---
Chelsea: 1.94 xG   |   Bournemouth: 1.07 xG

--- OUTCOME PROBABILITIES ---
Chelsea Win:   41.5%  ██████████
Draw:              23.8%  ██████
Bournemouth Win:   34.7%  █████████

--- TOP 5 MOST LIKELY SCOREBREAKDOWNS ---
  1 - 1   : 10.2%
  2 - 1   :  9.9%
  1 - 0   :  9.6%
  2 - 0   :  9.3%
  3 - 1   :  6.4%


MATCH FORECAST: IPSWICH (Home) vs FULHAM (Away)

--- TALE OF THE TAPE ---
                 Me